# Mega-Heracross — Topology Diversity Fix
**ITEMS 1–6: Multi-region GT masks, diverse training, held-out generalization test**

### Prerequisites
1. `colab_upload.zip` already in Google Drive from last session
2. Runtime → Change runtime type → **T4 GPU**
3. Run cells in order

In [ ]:
# ============================================================
# CELL 1: Mount Drive + setup dirs
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

ZIP_PATH = '/content/drive/MyDrive/colab_upload.zip'
WORK_DIR = '/content/mega-heracross'

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(WORK_DIR)

os.chdir(WORK_DIR)

for d in ['part_a_vision/models', 'part_a_vision/outputs/calibrated_training',
          'part_a_vision/data/koramangala/train', 'part_a_vision/data/koramangala/val',
          'outputs', 'region_gt_masks', 'region_tiles']:
    os.makedirs(d, exist_ok=True)

print(f'Ready: {WORK_DIR}')

In [ ]:
# ============================================================
# CELL 2: Install dependencies
# ============================================================
import subprocess, sys
for pkg in ['transformers>=4.37.0', 'timm', 'albumentations',
            'osmnx', 'opencv-python-headless', 'scipy',
            'scikit-image', 'sknw', 'shapely']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('All packages installed.')

In [ ]:
# ============================================================
# CELL 3: GPU + common imports
# ============================================================
import sys, os, time, csv
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
import osmnx as ox
from pathlib import Path
from scipy.ndimage import gaussian_filter

sys.path.insert(0, WORK_DIR)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU. Switch runtime to T4 GPU.')

In [ ]:
# ============================================================
# CELL 4: ITEM 1 — Download OSMnx graphs + rasterize GT masks
# ============================================================

TILE_SIZE = 512

# 10 Bengaluru regions: (lon_min, lat_min, lon_max, lat_max)
# 8 TRAINING + 2 HELD-OUT (never seen during training)
REGIONS = {
    # ── TRAINING REGIONS (8) ──
    'koramangala':  (77.6101, 12.9177, 77.6401, 12.9377),
    'hsr_layout':   (77.6350, 12.9000, 77.6650, 12.9200),
    'indiranagar':  (77.6300, 12.9650, 77.6600, 12.9850),
    'jayanagar':    (77.5700, 12.9200, 77.6000, 12.9400),
    'btm_layout':   (77.6050, 12.9000, 77.6350, 12.9200),
    'malleswaram':  (77.5550, 13.0050, 77.5850, 13.0250),
    'jp_nagar':     (77.5800, 12.8900, 77.6100, 12.9100),
    'rajajinagar':  (77.5350, 13.0050, 77.5650, 13.0250),
    # ── HELD-OUT TEST REGIONS (2, never used in training) ──
    'basavanagudi': (77.5650, 12.9350, 77.5950, 12.9550),
    'whitefield':   (77.7400, 12.9600, 77.7700, 12.9800),
}
TRAIN_REGIONS = ['koramangala','hsr_layout','indiranagar','jayanagar',
                 'btm_layout','malleswaram','jp_nagar','rajajinagar']
TEST_REGIONS  = ['basavanagudi', 'whitefield']

ROAD_WIDTH_PX = 2  # ~12m at 5.8m/px — standard 2-lane road


def rasterize_osmnx(G, bbox, tile_size=512, road_width=2):
    """
    Rasterize OSMnx graph edges to a binary road mask.
    Uses edge geometry if available, otherwise straight line between nodes.
    Returns (H, W) uint8 mask.
    """
    lon_min, lat_min, lon_max, lat_max = bbox
    mask = np.zeros((tile_size, tile_size), dtype=np.uint8)

    def geo_to_px(lat, lon):
        c = int(np.clip((lon - lon_min) / (lon_max - lon_min) * (tile_size - 1), 0, tile_size - 1))
        r = int(np.clip((lat_max - lat)  / (lat_max - lat_min) * (tile_size - 1), 0, tile_size - 1))
        return (c, r)  # (x, y) for cv2

    for u, v, data in G.edges(data=True):
        try:
            u_d = G.nodes[u]; v_d = G.nodes[v]
            if 'geometry' in data:
                coords = list(data['geometry'].coords)  # (lon, lat) pairs
                pts = [geo_to_px(lat, lon) for lon, lat in coords]
            else:
                pts = [geo_to_px(u_d['y'], u_d['x']),
                       geo_to_px(v_d['y'], v_d['x'])]
            for i in range(len(pts) - 1):
                cv2.line(mask, pts[i], pts[i+1], 1, road_width)
        except Exception:
            continue
    return mask


region_masks = {}   # region_name -> np.ndarray(512,512,uint8)
region_graphs = {}  # region_name -> nx graph (for pipeline eval)

print('ITEM 1: Downloading OSMnx + rasterizing GT masks')
print(f'{'Region':<16} {'Status':<12} {'Nodes':>7} {'Edges':>7} {'Road_px':>9} {'Density%':>9}')
print('-' * 65)

for rname, bbox in REGIONS.items():
    lon_min, lat_min, lon_max, lat_max = bbox
    center_lat = (lat_min + lat_max) / 2
    center_lon = (lon_min + lon_max) / 2

    try:
        G = ox.graph_from_point(
            (center_lat, center_lon), dist=1700,
            network_type='drive', simplify=True, retain_all=False
        )
        status = 'OK'
    except Exception as e:
        print(f'  {rname:<16} FAILED: {e}')
        continue

    mask = rasterize_osmnx(G, bbox, TILE_SIZE, ROAD_WIDTH_PX)

    # Auto-adjust width if density is out of 2-10% range
    density = mask.sum() / mask.size * 100
    if density > 10.0:  # too dense
        mask = rasterize_osmnx(G, bbox, TILE_SIZE, 1)
        density = mask.sum() / mask.size * 100
        status = 'OK(w=1)'
    elif density < 1.5:  # too sparse
        mask = rasterize_osmnx(G, bbox, TILE_SIZE, 3)
        density = mask.sum() / mask.size * 100
        status = 'OK(w=3)'

    region_masks[rname]  = mask
    region_graphs[rname] = G

    np.save(f'region_gt_masks/{rname}.npy', mask)

    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    print(f'  {rname:<16} {status:<12} {n_nodes:>7} {n_edges:>7} '
          f'{int(mask.sum()):>9,} {density:>8.2f}%')

print(f'\nLoaded {len(region_masks)} regions.')
train_densities = [region_masks[r].sum()/region_masks[r].size*100
                   for r in TRAIN_REGIONS if r in region_masks]
test_densities  = [region_masks[r].sum()/region_masks[r].size*100
                   for r in TEST_REGIONS  if r in region_masks]
print(f'Train region density: mean={np.mean(train_densities):.2f}%  '
      f'min={min(train_densities):.2f}%  max={max(train_densities):.2f}%')
print(f'Test  region density: mean={np.mean(test_densities):.2f}%  '
      f'min={min(test_densities):.2f}%  max={max(test_densities):.2f}%')

In [ ]:
# ============================================================
# CELL 5: Visualize GT mask diversity
# ============================================================
n_regions = len(region_masks)
ncols = 5
nrows = (n_regions + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*4))
axes = axes.flatten()

for i, (rname, mask) in enumerate(region_masks.items()):
    density = mask.sum() / mask.size * 100
    tag = '[TRAIN]' if rname in TRAIN_REGIONS else '[HELD-OUT]'
    axes[i].imshow(mask, cmap='gray')
    axes[i].set_title(f'{rname}\n{tag}  {density:.2f}%', fontsize=9)
    axes[i].axis('off')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle('ITEM 1: GT Mask Diversity — 10 Bengaluru Regions', fontsize=13)
plt.tight_layout()
plt.savefig('part_a_vision/outputs/calibrated_training/item1_gt_mask_diversity.png',
            dpi=100, bbox_inches='tight')
plt.show()
print('Saved: item1_gt_mask_diversity.png')

In [ ]:
# ============================================================
# CELL 6: ITEM 2 — RegionDiverseDataset
# 96 train tiles (8 regions x 12 tiles each)
# 24 val tiles   (8 regions x 3 tiles each, in-distribution)
# 20 test tiles  (2 held-out regions x 10 tiles each)
# ============================================================
from scipy.ndimage import gaussian_filter

N_OPT  = 4  # optical bands: Green, Red, NIR, SWIR
N_SAR  = 2  # SAR bands: VV, VH


def make_diverse_tile(gt_mask, seed, tile_size=512):
    """
    Generate a synthetic 12-channel fused tile consistent with gt_mask.
    Returns (fused: float32 (12,H,W), mask: int64 (H,W))
    """
    rng = np.random.RandomState(seed)
    H = W = tile_size

    # ── Optical (4 bands: Green, Red, NIR, SWIR) ──
    optical = np.zeros((N_OPT, H, W), dtype=np.float32)
    for b in range(N_OPT):
        base = rng.uniform(0.15, 0.45, (H, W)).astype(np.float32)
        optical[b] = gaussian_filter(base, sigma=rng.uniform(3, 12))

    # ── SAR (2 bands: VV, VH) ──
    sar = np.zeros((N_SAR, H, W), dtype=np.float32)
    for b in range(N_SAR):
        sar[b] = rng.rayleigh(scale=0.3, size=(H, W)).astype(np.float32)
        sar[b] = gaussian_filter(sar[b], sigma=1.0)
        lo, hi = np.percentile(sar[b], [1, 99])
        sar[b] = np.clip((sar[b] - lo) / max(hi - lo, 1e-8), 0, 1)

    # ── Paint roads ──
    road_px = (gt_mask == 1)
    n_road  = int(road_px.sum())
    if n_road > 0:
        for b in range(N_OPT):
            optical[b][road_px] = rng.uniform(0.05, 0.2, n_road)
        for b in range(N_SAR):
            sar[b][road_px] = rng.uniform(0.0, 0.15, n_road)

    # ── Buildings (bright SAR) ──
    for _ in range(rng.randint(8, 25)):
        cx, cy = rng.randint(W//8, 7*W//8), rng.randint(H//8, 7*H//8)
        bw, bh = rng.randint(3,10), rng.randint(3,10)
        x0,x1 = max(0,cx-bw//2), min(W,cx+bw//2)
        y0,y1 = max(0,cy-bh//2), min(H,cy+bh//2)
        for b in range(N_SAR): sar[b][y0:y1,x0:x1] = rng.uniform(0.7,1.0)

    # ── Normalize optical ──
    for b in range(N_OPT):
        lo, hi = np.percentile(optical[b], [2, 98])
        optical[b] = np.clip((optical[b]-lo)/max(hi-lo,1e-8), 0, 1)

    # ── Spectral indices (NDVI, NDWI) ──
    nir, red, green = optical[2], optical[1], optical[0]
    ndvi = (nir - red)   / (nir + red   + 1e-6)
    ndwi = (green - nir) / (green + nir + 1e-6)
    ndvi = ((ndvi + 1.0) / 2.0)[None]  # (1,H,W)
    ndwi = ((ndwi + 1.0) / 2.0)[None]

    # ── Temporal diff (zeros — no temporal pair available) ──
    temporal = np.zeros((4, H, W), dtype=np.float32)

    # ── Concat: 4 opt + 2 idx + 2 SAR + 4 temporal = 12 channels ──
    fused = np.concatenate([optical, ndvi, ndwi, sar, temporal], axis=0)  # (12,H,W)
    return fused.astype(np.float32), gt_mask.astype(np.int64)


class RegionDiverseDataset(torch.utils.data.Dataset):
    """
    Dataset that samples across multiple region GT masks.
    Each tile picks a region deterministically from its seed,
    ensuring even coverage across all supplied regions.
    """
    def __init__(self, region_names, region_mask_dict, n_tiles, seed_offset=0, tile_size=512):
        self.region_names    = region_names
        self.region_mask_dict = region_mask_dict
        self.n_tiles         = n_tiles
        self.seed_offset     = seed_offset
        self.tile_size       = tile_size
        # Pre-assign regions round-robin so every region gets equal tiles
        self.tile_regions    = [region_names[i % len(region_names)] for i in range(n_tiles)]

    def __len__(self):
        return self.n_tiles

    def __getitem__(self, idx):
        seed    = self.seed_offset + idx
        rname   = self.tile_regions[idx]
        gt_mask = self.region_mask_dict[rname]
        fused, mask = make_diverse_tile(gt_mask, seed, self.tile_size)
        return torch.from_numpy(fused), torch.from_numpy(mask)


# Build datasets
avail_train = [r for r in TRAIN_REGIONS if r in region_masks]
avail_test  = [r for r in TEST_REGIONS  if r in region_masks]

N_TRAIN = len(avail_train) * 12  # 96 tiles
N_VAL   = len(avail_train) * 3   # 24 tiles
N_TEST  = len(avail_test)  * 10  # 20 tiles

train_ds = RegionDiverseDataset(avail_train, region_masks, N_TRAIN, seed_offset=0)
val_ds   = RegionDiverseDataset(avail_train, region_masks, N_VAL,   seed_offset=10000)
test_ds  = RegionDiverseDataset(avail_test,  region_masks, N_TEST,  seed_offset=20000)

print('ITEM 2: Diverse dataset built')
print(f'  Train: {N_TRAIN} tiles across {len(avail_train)} regions')
print(f'  Val:   {N_VAL}   tiles across {len(avail_train)} regions (in-distribution)')
print(f'  Test:  {N_TEST}  tiles across {len(avail_test)}  regions (HELD-OUT, never seen)')
print(f'  Training regions:  {avail_train}')
print(f'  Held-out regions:  {avail_test}')

# Verify density distribution
sample_densities = {'train': [], 'val': [], 'test': []}
for split, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    for i in range(min(8, len(ds))):
        _, mask = ds[i]
        sample_densities[split].append(mask.float().mean().item() * 100)

print()
print('ITEM 2: Road density verification (8-tile sample per split):')
for split, d in sample_densities.items():
    print(f'  {split:>5}: mean={np.mean(d):.2f}%  min={min(d):.2f}%  max={max(d):.2f}%')
print('  (Target: 3-8%. Calibration held from last session.)')

In [ ]:
# ============================================================
# CELL 7: ITEM 3 — Retrain 20 epochs on diverse dataset
# ============================================================
from part_a_vision.model import build_model
from part_a_vision.loss import CombinedLoss

EPOCHS     = 20
BATCH_SIZE = 4
LR         = 1e-4
POS_WEIGHT = 27.0   # unchanged from last session
CKPT_PATH  = Path('part_a_vision/models/best_checkpoint_diverse.pth')
OUT_DIR    = Path('part_a_vision/outputs/calibrated_training')

print('=' * 65)
print('ITEM 3: DIVERSE TRAINING (20 epochs, SegFormer B3, T4)')
print(f'  Regions: {avail_train}')
print(f'  Train tiles: {N_TRAIN} | Val tiles: {N_VAL}')
print(f'  pos_weight: {POS_WEIGHT}')
print('=' * 65)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(
    val_ds,   batch_size=BATCH_SIZE, shuffle=False,num_workers=2, pin_memory=True)

model = build_model(backbone='segformer_b3', input_channels=12, num_classes=1)
model = model.to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: segformer_b3 | {n_params:,} params')

criterion = CombinedLoss(
    dice_weight=0.4, bce_weight=0.3, boundary_weight=0.2, conn_weight=0.1,
    pos_weight=POS_WEIGHT, use_focal=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.01)
scaler    = torch.amp.GradScaler('cuda', enabled=(device=='cuda'))

def compute_metrics(logits, targets, threshold=0.5):
    probs   = torch.sigmoid(logits)
    preds   = (probs > threshold).long()
    targets = targets.long()
    tp = (preds * targets).sum().float()
    fp = (preds * (1 - targets)).sum().float()
    fn = ((1 - preds) * targets).sum().float()
    union  = (preds + targets).clamp(0, 1).sum().float()
    iou    = (tp / (union + 1e-8)).item()
    prec   = (tp / (tp + fp + 1e-8)).item()
    rec    = (tp / (tp + fn + 1e-8)).item()
    f1     = 2 * prec * rec / (prec + rec + 1e-8)
    return {'iou': iou, 'precision': prec, 'recall': rec, 'f1': f1}

best_iou, best_epoch = 0.0, 0
train_log = []

hdr = f'{"Ep":>3} {"TrLoss":>9} {"TrIoU":>8} {"VaLoss":>9} {"VaIoU":>8} {"VaPrec":>8} {"VaRec":>7} {"VaF1":>7} {"Time":>7}'
print(f'\n{hdr}')
print('-' * len(hdr))

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    model.train()
    tl, ti, n = 0., 0., 0
    for fused, mask in train_loader:
        fused = fused.to(device, non_blocking=True)
        mask  = mask.to(device,  non_blocking=True).float().unsqueeze(1)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device=='cuda')):
            out = model(fused)
            if isinstance(out, dict): out = out['out']
            loss, _ = criterion(out, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tl += loss.item(); ti += compute_metrics(out.detach(), mask.squeeze(1))['iou']; n += 1
    scheduler.step()
    tl /= max(n,1); ti /= max(n,1)

    model.eval()
    vl, vm, n = 0., [], 0
    with torch.no_grad():
        for fused, mask in val_loader:
            fused = fused.to(device, non_blocking=True)
            mask  = mask.to(device,  non_blocking=True).float().unsqueeze(1)
            out   = model(fused)
            if isinstance(out, dict): out = out['out']
            loss, _ = criterion(out, mask)
            vl += loss.item(); vm.append(compute_metrics(out, mask.squeeze(1))); n += 1
    vl  /= max(n,1)
    vi   = np.mean([m['iou']       for m in vm])
    vp   = np.mean([m['precision'] for m in vm])
    vr   = np.mean([m['recall']    for m in vm])
    vf   = np.mean([m['f1']        for m in vm])
    elapsed = time.time() - t0

    is_best = vi > best_iou
    if is_best:
        best_iou, best_epoch = vi, epoch
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'best_val_iou': best_iou}, CKPT_PATH)

    flag = ' **BEST**' if is_best else ''
    print(f'{epoch:>3} {tl:>9.4f} {ti:>8.4f} {vl:>9.4f} {vi:>8.4f} {vp:>8.4f} {vr:>7.4f} {vf:>7.4f} {elapsed:>6.1f}s{flag}')
    train_log.append(dict(epoch=epoch, train_loss=tl, train_iou=ti,
                          val_loss=vl, val_iou=vi, val_prec=vp, val_rec=vr, val_f1=vf))

print('-' * len(hdr))
print(f'DONE. Best val IoU = {best_iou:.4f} at epoch {best_epoch}')

In [ ]:
# ============================================================
# CELL 8: ITEM 4 — Held-out region evaluation (generalization test)
# Regions: basavanagudi, whitefield — NEVER seen during training
# ============================================================
print('=' * 65)
print('ITEM 4: HELD-OUT REGION EVALUATION (true generalization test)')
print(f'  Held-out regions: {avail_test}')
print('=' * 65)

test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=4, shuffle=False, num_workers=2)

model.eval()
heldout_metrics = {r: [] for r in avail_test}

with torch.no_grad():
    for tile_idx in range(len(test_ds)):
        fused_t, mask_t = test_ds[tile_idx]
        rname = test_ds.tile_regions[tile_idx]
        out = model(fused_t.unsqueeze(0).to(device))
        if isinstance(out, dict): out = out['out']
        m = compute_metrics(out, mask_t.to(device).float())
        heldout_metrics[rname].append(m)

print(f'\n{"Region":<16} {"N":>4} {"IoU":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}')
print('-' * 58)

all_heldout_iou = []
for rname in avail_test:
    ms = heldout_metrics[rname]
    if not ms:
        print(f'  {rname:<16} no tiles')
        continue
    iou  = np.mean([m['iou']       for m in ms])
    prec = np.mean([m['precision'] for m in ms])
    rec  = np.mean([m['recall']    for m in ms])
    f1   = np.mean([m['f1']        for m in ms])
    all_heldout_iou.append(iou)
    print(f'  {rname:<16} {len(ms):>4} {iou:>8.4f} {prec:>8.4f} {rec:>8.4f} {f1:>8.4f}')

held_iou_mean = np.mean(all_heldout_iou)
val_iou_final = train_log[-1]['val_iou']
generalization_gap = val_iou_final - held_iou_mean

print(f'\n  In-distribution val IoU:   {val_iou_final:.4f}')
print(f'  Held-out region IoU (mean): {held_iou_mean:.4f}')
print(f'  Generalization gap:         {generalization_gap:.4f}')

if held_iou_mean >= 0.30:
    gen_verdict = 'PARTIAL — model generalizes to unseen layouts at IoU>=0.30'
elif held_iou_mean >= 0.15:
    gen_verdict = 'WEAK — some generalization but significant gap to in-distribution'
else:
    gen_verdict = 'NO — model does not generalize to unseen road layouts'
print(f'  Generalization verdict:     {gen_verdict}')

In [ ]:
# ============================================================
# CELL 9: ITEM 5 — Full pipeline eval on Koramangala OSM F1
# Compares against the fixed OSM reference (2708 nodes, 6767 edges)
# ============================================================
import sknw
from skimage.morphology import skeletonize

print('=' * 65)
print('ITEM 5: PIPELINE EVAL — Part A -> Part B -> OSM F1')
print('  Reference: Koramangala OSM (2708 nodes, 6767 edges)')
print('=' * 65)

KORAMANGALA_BBOX = (77.6101, 12.9177, 77.6401, 12.9377)

# Part A: inference on a Koramangala tile
kora_mask = region_masks.get('koramangala')
if kora_mask is None:
    kora_mask = np.load('part_a_vision/data/koramangala/osmnx_gt_mask.npy')

fused_k, gt_k = make_diverse_tile(kora_mask, seed=99999)
model.eval()
with torch.no_grad():
    out_k = model(torch.from_numpy(fused_k).unsqueeze(0).to(device))
    if isinstance(out_k, dict): out_k = out_k['out']
    prob_k = torch.sigmoid(out_k).squeeze().cpu().numpy()

pred_mask_k = (prob_k > 0.5).astype(np.uint8)
pred_pct    = pred_mask_k.sum() / pred_mask_k.size * 100
gt_pct      = gt_k.sum() / gt_k.size * 100
m_a         = compute_metrics(out_k, torch.from_numpy(gt_k).float().to(device))

print(f'\nPart A (Koramangala tile):')
print(f'  GT density:   {gt_pct:.2f}%')
print(f'  Pred density: {pred_pct:.2f}%')
print(f'  Part A IoU:   {m_a["iou"]:.4f}')

# Part B: skeletonization
skel   = skeletonize(pred_mask_k > 0).astype(np.uint8)
G_skel = sknw.build_sknw(skel)
n_nodes_pred = G_skel.number_of_nodes()
n_edges_pred = G_skel.number_of_edges()
print(f'\nPart B skeleton: {n_nodes_pred} nodes, {n_edges_pred} edges')

# Geo conversion
def px_to_geo(row, col, bbox=KORAMANGALA_BBOX, sz=512):
    lat = bbox[3] - (row / sz) * (bbox[3] - bbox[1])
    lon = bbox[0] + (col / sz) * (bbox[2] - bbox[0])
    return lat, lon

pred_node_coords = []
for nid, d in G_skel.nodes(data=True):
    r, c = d['o']
    pred_node_coords.append(px_to_geo(r, c))

pred_edge_coords = []
for u, v, d in G_skel.edges(data=True):
    pts = d.get('pts', [])
    if len(pts):
        mid = pts[len(pts)//2]
        pred_edge_coords.append(px_to_geo(mid[0], mid[1]))

# Download Koramangala OSM reference
print('\nDownloading Koramangala OSM reference...')
G_osm = ox.graph_from_point((12.9277, 77.6251), dist=1700,
                              network_type='drive', simplify=True)
osm_node_coords = [(d['y'], d['x']) for _, d in G_osm.nodes(data=True)]
osm_edge_coords = []
for u, v, d in G_osm.edges(data=True):
    ud, vd = G_osm.nodes[u], G_osm.nodes[v]
    osm_edge_coords.append(((ud['y']+vd['y'])/2, (ud['x']+vd['x'])/2))
print(f'  OSM: {len(osm_node_coords)} nodes, {len(osm_edge_coords)} edges')

# Spatial matching (30m threshold)
THRESH = 0.00027
def match(pred, ref, thresh):
    mr = set()
    tp = 0
    for plat, plon in pred:
        for ri, (rlat, rlon) in enumerate(ref):
            if ri in mr: continue
            if ((plat-rlat)**2 + (plon-rlon)**2)**0.5 < thresh:
                tp += 1; mr.add(ri); break
    fp = len(pred) - tp
    fn = len(ref)  - len(mr)
    pr = tp / (tp+fp+1e-8); re = tp / (tp+fn+1e-8)
    return pr, re, 2*pr*re/(pr+re+1e-8), tp, fp, fn

np_, nr_, nf1_, ntp, nfp, nfn = match(pred_node_coords, osm_node_coords, THRESH)
ep_, er_, ef1_, etp, efp, efn = match(pred_edge_coords, osm_edge_coords, THRESH)

print(f'\n[ITEM 5 RESULTS — Koramangala OSM F1]')
print(f'  Pred nodes: {n_nodes_pred:>6} | OSM nodes: {len(osm_node_coords):>6}')
print(f'  Pred edges: {n_edges_pred:>6} | OSM edges: {len(osm_edge_coords):>6}')
print(f'  Node F1:  {nf1_:.4f}  (Prec={np_:.4f}, Rec={nr_:.4f})  TP={ntp} FP={nfp} FN={nfn}')
print(f'  Edge F1:  {ef1_:.4f}  (Prec={ep_:.4f}, Rec={er_:.4f})  TP={etp} FP={efp} FN={efn}')
print(f'  [BASELINE last session] Node F1=0.0166, Edge F1=0.0088')

In [ ]:
# ============================================================
# CELL 10: ITEM 6 — Final honest report
# ============================================================
final = train_log[-1]

all_densities = [region_masks[r].sum()/region_masks[r].size*100
                 for r in region_masks]

print('=' * 65)
print('ITEM 6: FINAL HONEST REPORT')
print('=' * 65)
print()
print('TRAINING REGIONS USED:', avail_train)
print('HELD-OUT TEST REGIONS:', avail_test)
print()
print('DATASET ROAD DENSITY:')
for r in region_masks:
    d = region_masks[r].sum()/region_masks[r].size*100
    tag = '[TRAIN]' if r in TRAIN_REGIONS else '[HELD-OUT]'
    print(f'  {r:<18} {d:.2f}% {tag}')
print(f'  Overall mean={np.mean(all_densities):.2f}%  '
      f'min={min(all_densities):.2f}%  max={max(all_densities):.2f}%')
print()
print(f'EPOCHS TRAINED: {EPOCHS}')
print()
print('TRAINING TABLE (all epochs):')
print(f'{"Ep":>3} {"TrLoss":>9} {"TrIoU":>8} {"VaLoss":>9} {"VaIoU":>8} {"VaPrec":>8} {"VaRec":>7} {"VaF1":>7}')
for row in train_log:
    print(f'{row["epoch"]:>3} {row["train_loss"]:>9.4f} {row["train_iou"]:>8.4f} '
          f'{row["val_loss"]:>9.4f} {row["val_iou"]:>8.4f} '
          f'{row["val_prec"]:>8.4f} {row["val_rec"]:>7.4f} {row["val_f1"]:>7.4f}')
print()
print(f'VAL IoU (in-distribution):       LAST SESSION={0.5905:.4f} -> THIS SESSION={val_iou_final:.4f}')
print(f'VAL F1  (in-distribution):       LAST SESSION={0.7425:.4f} -> THIS SESSION={final["val_f1"]:.4f}')
print()
print(f'HELD-OUT REGION IoU (true generalization test):')
for rname in avail_test:
    ms = heldout_metrics[rname]
    if ms:
        iou = np.mean([m['iou'] for m in ms])
        f1  = np.mean([m['f1']  for m in ms])
        print(f'  {rname}: IoU={iou:.4f}, F1={f1:.4f}')
print(f'  Mean held-out IoU: {held_iou_mean:.4f}')
print(f'  Generalization gap (val IoU - held-out IoU): {generalization_gap:.4f}')
print()
print(f'PART B NODE F1:   LAST SESSION=0.0166 -> THIS SESSION={nf1_:.4f}')
print(f'  Node Prec: {np_:.4f} | Node Rec: {nr_:.4f}')
print(f'  Pred nodes: {n_nodes_pred} | OSM nodes: {len(osm_node_coords)}')
print(f'PART B EDGE F1:   LAST SESSION=0.0088 -> THIS SESSION={ef1_:.4f}')
print(f'  Edge Prec: {ep_:.4f} | Edge Rec: {er_:.4f}')
print(f'  Pred edges: {n_edges_pred} | OSM edges: {len(osm_edge_coords)}')
print()
print(f'IS THIS MODEL GENERALIZING TO UNSEEN LAYOUTS: {gen_verdict}')
print()
print('REMAINING GAPS:')
print('  1. All training data is synthetic (optical+SAR is procedurally generated).')
print('     Real satellite tiles would expose further domain gaps.')
print('  2. Node recall is limited by the 512x512 tile covering ~7km2 but the')
print('     model cannot predict nodes far from learned patterns.')
print('  3. Temporal channels (channels 8-11) are zero throughout — not exploited.')
print('  4. Road width is fixed at 2px per region; real roads vary 1-6px by type.')
print('  5. No real satellite data tested (LISS-IV/Sentinel-1 not yet downloaded).')
print('=' * 65)

In [ ]:
# ============================================================
# CELL 11: Save all artifacts to Drive
# ============================================================
import shutil
DRIVE_OUT = '/content/drive/MyDrive/mega-heracross-colab-output'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Checkpoint
if CKPT_PATH.exists():
    shutil.copy(str(CKPT_PATH), DRIVE_OUT)
    print(f'Checkpoint: {DRIVE_OUT}/best_checkpoint_diverse.pth')

# Region GT masks
masks_dir = f'{DRIVE_OUT}/region_gt_masks'
os.makedirs(masks_dir, exist_ok=True)
for rname, mask in region_masks.items():
    np.save(f'{masks_dir}/{rname}.npy', mask)
print(f'GT masks: {masks_dir}/')

# Diversity visualization
vis = OUT_DIR / 'item1_gt_mask_diversity.png'
if vis.exists(): shutil.copy(str(vis), DRIVE_OUT)

# Train log CSV
csv_path = f'{DRIVE_OUT}/train_log_diverse.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=train_log[0].keys())
    w.writeheader(); w.writerows(train_log)
print(f'Train log: {csv_path}')

print('\nAll artifacts saved to Drive.')